# no-relu-on-final-layer — worked example 3: bare final layer in a tiny generator

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `no-relu-on-final-layer`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Decoders and generators stack activation-bearing blocks but leave the LAST layer bare so the output can take any real value (e.g. image pixels in [-1, 1] after a tanh, or unbounded features). Copy-pasting a block pattern onto the final layer is the classic mistake.

## Worked solution

We build `TinyGenerator` as an `nn.Sequential` of two upsampling-style ConvTranspose blocks. Each intermediate block is Conv-transpose followed by ReLU, but the final ConvTranspose2d has NO activation after it, so the generated output is free to be negative. We seed, push a small latent map through, and report the fraction of negative pixels in the output, which is substantial, confirming the final layer is bare. Printing the output shape and the negative fraction shows the generator produces full-range values.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(2)

class TinyGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.up1 = nn.ConvTranspose2d(4, 8, kernel_size=4, stride=2, padding=1)
        self.up2 = nn.ConvTranspose2d(8, 1, kernel_size=4, stride=2, padding=1)
    def forward(self, z):
        h = t.relu(self.up1(z))   # block with activation
        return self.up2(h)        # FINAL layer: bare, no activation

gen = TinyGenerator()
z = t.randn(2, 4, 4, 4)
out = gen(z)
frac_neg = (out < 0).float().mean().item()
print('output shape:', tuple(out.shape))
print('fraction negative:', round(frac_neg, 3))